In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email() -> str:
    """Read the email from the current state. ALWAYS call this first before responding."""
    # Access email from state via the agent context
    return "EMAIL_PLACEHOLDER"  # Will be overridden by state injection

@tool  
def send_email(body: str, recipient: str = "John") -> str:
    """Send an email response. Requires human approval before execution."""
    return f"✅ Email sent to {recipient}: {body}"

In [4]:
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailAgentState(AgentState):
    messages: Annotated[list, add_messages]
    email: str


def create_email_tools(email: str):
    @tool
    def read_email() -> str:
        """Read the email from the current state."""
        return email
    
    @tool  
    def send_email(body: str, recipient: str = "John") -> str:
        """Send an email response. Requires human approval."""
        return f"✅ Email sent to {recipient}: {body}"
    
    return [read_email, send_email]


# Get email from input
email_content = "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."

agent = create_agent(
    model=model,
    tools=create_email_tools(email_content),  # Tools with email injected
    state_schema=EmailAgentState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="⚠️ Tool execution requires approval",
        ),
    ],
)

In [5]:
from pprint import pprint
from langchain_core.messages import HumanMessage, SystemMessage

config = {"configurable": {"thread_id": "1"}}


input_data = {
    "messages": [
        SystemMessage(content="You are an email assistant. Read the email first, then draft and send a response."),
        HumanMessage(content="Please read my email and send a response.")
    ]
    # Note: email is now in the tools, not state
}

print("🚀 Starting agent execution...\n")

for event in agent.stream(input_data, config=config, stream_mode="updates"):
    pprint(event)
    print("-" * 60)

🚀 Starting agent execution...

{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-15T22:46:38.117958719Z', 'done': True, 'done_reason': 'stop', 'total_duration': 336951478740, 'load_duration': 29389213678, 'prompt_eval_count': 198, 'prompt_eval_duration': 48372962736, 'eval_count': 488, 'eval_duration': 258771232554, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cf3a8-f0e1-7230-831c-ae97249e65b6-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'e2e91033-5e9b-4917-a16a-1b04f4496a29', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 488, 'total_tokens': 686})]}}
------------------------------------------------------------
{'HumanInTheLoopMiddleware.after_model': None}
------------------------------------------------------------
{'tools': {'messages': [ToolMessage(content="Hi Seán, I'm going to be late for ou

In [6]:
state = agent.get_state(config)
interrupts = state.metadata.get("interrupts", [])

if interrupts:
    print("\n⚠️  EXECUTION INTERRUPTED!")
    print(f"Interrupt details: {interrupts}")
    
    print("\n📋 Current State:")
    pprint(state.values)
    
    # Resume execution after human approval
    print("\n✅ Resuming execution after approval...\n")
    for event in agent.stream(None, config=config, stream_mode="updates"):
        pprint(event)
        print("-" * 60)
else:
    print("\n✅ Execution completed without interrupts")


✅ Execution completed without interrupts


In [7]:
final_state = agent.get_state(config)
print("\n📬 FINAL STATE:")
pprint(final_state.values)


📬 FINAL STATE:
{'messages': [SystemMessage(content='You are an email assistant. Read the email first, then draft and send a response.', additional_kwargs={}, response_metadata={}, id='f226ce67-75dc-4a35-8929-869ab2cfaacc'),
              HumanMessage(content='Please read my email and send a response.', additional_kwargs={}, response_metadata={}, id='2a92cb72-38a9-48ff-aef6-77eea5fbaee3'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-15T22:46:38.117958719Z', 'done': True, 'done_reason': 'stop', 'total_duration': 336951478740, 'load_duration': 29389213678, 'prompt_eval_count': 198, 'prompt_eval_duration': 48372962736, 'eval_count': 488, 'eval_duration': 258771232554, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cf3a8-f0e1-7230-831c-ae97249e65b6-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'e2e91033-5e9b-4917-a16a-1b04f4496a29', 'type': 'tool_call'}], inval